---

## Step 1: Install the AgentCore CLI

In [ ]:
%pip install bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents boto3

---

## Step 2: Set up execution roles

Creates the IAM execution roles for the AgentCore runtimes (idempotent — safe to re-run).

In [ ]:
import sys, os, boto3
sys.path.insert(0, "../../shared")
import deploy_utils as u

session = u.get_session()
account = u.get_account(session)
bucket  = u.code_bucket_name(account, u.REGION)
iam     = session.client("iam", region_name=u.REGION)

# Create specialist runtime role (Bedrock, Logs, S3) — shared by researcher/analyzer/critic_refiner
runtime_arn = u.ensure_runtime_role(
    iam, f"workshop-agentcore-m8-runtime-role",
    account, u.REGION, bucket,
)
os.environ["AGENTCORE_RUNTIME_ROLE_ARN"] = runtime_arn
print(f"Runtime role: {runtime_arn}")

# NOTE: the orchestrator role is created by deploy.py AFTER the Memory resource exists,
# so it can include the memory permissions. Do NOT pre-create it here.

---

## Step 3: Deploy

Runs all four runtimes + AgentCore Memory (~5-8 min). The cell below executes `deploy.py` directly so it picks up the role ARNs set in the previous cell.

In [ ]:
!python deploy.py --name-prefix m8

In [ ]:
import os
from pathlib import Path

_candidates = [
    Path("/workshop/samples/08-capstone/production/.runtime_arn"),
    Path(".runtime_arn"),
]
_arn_path = next((p for p in _candidates if p.exists()), None)
if _arn_path is None:
    raise FileNotFoundError("Cannot find .runtime_arn — run the deploy step first.")

RUNTIME_ARN = _arn_path.read_text().strip()
print(f"Reading from: {_arn_path}")
print(f"RUNTIME_ARN = {RUNTIME_ARN}")

---

## Step 4: Invoke the deployed agent

The `RUNTIME_ARN` was captured from the deploy output above. The cell below runs a single test invocation. To run a multi-turn conversation, run the second cell below it.

In [ ]:
import os, boto3, json, uuid
from botocore.config import Config

REGION = os.environ.get("AWS_REGION", "us-east-1")

if not RUNTIME_ARN:
    raise ValueError("RUNTIME_ARN not set — re-run Step 2 (Deploy).")

client = boto3.client(
    "bedrock-agentcore",
    region_name=REGION,
    config=Config(read_timeout=300),
)

response = client.invoke_agent_runtime(
    agentRuntimeArn=RUNTIME_ARN,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({
        "prompt": "NovaCart Premium Tier: Options A ($19.99/mo invite-only), B ($14.99/mo 5% pilot), C ($12.99/mo full launch). Target: +15% CLV in 6 months."
    }).encode(),
    qualifier="DEFAULT",
)

raw = response["response"].read()
try:
    result = json.loads(raw)
    print(result.get("response", result) if isinstance(result, dict) else result)
except Exception:
    print(raw.decode())

---

## 💬 Multi-turn chat

Run in a **new terminal** (interactive CLI — freezes if run in notebook):

```bash
python chat.py --actor-id workshop-user-01 --runtime-arn $(cat .runtime_arn)
```

> Change `workshop-user-01` to your alias. Press **Enter on an empty line** to exit.

---

## Step 5: Observability

After invoking, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- Root span per `invoke_agent_runtime` call
- Child span per `Agent()` call inside the pipeline
- Tool call spans nested under each agent
- Duration breakdown per stage

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore installs it automatically.

---

## Step 6: Cleanup

Uncomment and run the cell below to delete all AWS resources created by this module.

In [ ]:
# Uncomment and run to delete all resources created by this module.

# !python cleanup.py --name-prefix m8

# !python cleanup.py --name-prefix {prefix} --skip-memory
# Verify:
# import boto3, os
# REGION = os.environ.get("AWS_REGION", "us-east-1")
# remaining = boto3.client("bedrock-agentcore-control", region_name=REGION).list_agent_runtimes()
# print([rt["agentRuntimeName"] for rt in remaining.get("agentRuntimes", [])])